# Dialforge Cloud Benchmark v5

Commit-pinned, cache-resistant launcher for the Dialforge local AI benchmark. It tests **Qwen 3 1.7B, 4B and 8B**, **faster-whisper small.en**, **Chatterbox Nano**, and the full **STT → LLM → TTS** pipeline for all three Qwen tiers.

1. Choose **Runtime → Change runtime type → T4 GPU**.
2. Click **Runtime → Run all**.
3. Leave the tab open until the report appears.

**Launcher version: v5-commit-api.** This version resolves GitHub `main` to a commit SHA, fetches the bootstrap through the GitHub Contents API at that immutable SHA, validates that the bootstrap is v4/venv-v4, then streams every line live.

In [ ]:
# DIALFORGE BENCHMARK v5-commit-api
import base64, json, pathlib, shutil, subprocess, sys, time, urllib.request
from IPython.display import HTML, display

LAUNCHER_VERSION = 'v5-commit-api'
OWNER = 'SumamaAhmed69'
REPO = 'Axemetric-Caller-Beta-Runtime'
BOOTSTRAP_PATH = 'benchmarks/dialforge_colab_bootstrap.py'
BOOTSTRAP = pathlib.Path('/content/dialforge_colab_bootstrap.py')
LOG = pathlib.Path('/content/dialforge-bootstrap.log')
REPORT = pathlib.Path('/content/dialforge-benchmark/dialforge-benchmark-report.html')

def api_json(url):
    req = urllib.request.Request(url, headers={
        'User-Agent': 'Dialforge-Colab-v5',
        'Accept': 'application/vnd.github+json',
        'Cache-Control': 'no-cache',
        'Pragma': 'no-cache',
    })
    with urllib.request.urlopen(req, timeout=60) as response:
        return json.loads(response.read().decode('utf-8'))

print('Dialforge launcher:', LAUNCHER_VERSION, flush=True)
print('Resolving GitHub main to an immutable commit...', flush=True)
last_error = None
payload = None
resolved_sha = None
for attempt in range(1, 6):
    try:
        nonce = time.time_ns()
        commit = api_json(f'https://api.github.com/repos/{OWNER}/{REPO}/commits/main?dialforge={nonce}')
        resolved_sha = commit['sha']
        print('Resolved main commit:', resolved_sha, flush=True)
        item = api_json(f'https://api.github.com/repos/{OWNER}/{REPO}/contents/{BOOTSTRAP_PATH}?ref={resolved_sha}&dialforge={nonce}')
        if item.get('encoding') != 'base64' or not item.get('content'):
            raise RuntimeError('GitHub Contents API did not return base64 bootstrap content')
        payload = base64.b64decode(item['content'])
        text = payload.decode('utf-8')
        if not text.startswith('#!/usr/bin/env python3'):
            raise RuntimeError('Bootstrap did not have the expected Python header')
        if 'bootstrap v4' not in text:
            raise RuntimeError('Refusing stale bootstrap: expected marker bootstrap v4')
        if 'dialforge-benchmark-venv-v4' not in text:
            raise RuntimeError('Refusing stale bootstrap: expected venv-v4 path')
        if 'dialforge-benchmark-venv-v3' in text:
            raise RuntimeError('Refusing stale bootstrap: venv-v3 marker found')
        if 'python3-virtualenv' not in text or 'venv_is_healthy' not in text:
            raise RuntimeError('Refusing incomplete bootstrap: virtualenv safeguards are missing')
        compile(text, str(BOOTSTRAP), 'exec')
        BOOTSTRAP.write_bytes(payload)
        print(f'Bootstrap verified from commit {resolved_sha[:12]}: {len(payload)} bytes', flush=True)
        break
    except Exception as exc:
        last_error = exc
        print(f'GitHub API fetch attempt {attempt}/5 failed: {type(exc).__name__}: {exc}', flush=True)
        time.sleep(2 * attempt)
else:
    raise RuntimeError(f'Could not fetch a verified current Dialforge bootstrap: {last_error}')

# Clean only the obsolete half-created v3 environment from previous failed runs.
old_venv = pathlib.Path('/content/dialforge-benchmark-venv-v3')
if old_venv.exists():
    print('Removing obsolete broken v3 environment...', flush=True)
    shutil.rmtree(old_venv, ignore_errors=True)

print('\nStarting verified Dialforge bootstrap with LIVE output...', flush=True)
tail = []
with LOG.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(
        [sys.executable, '-u', str(BOOTSTRAP)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
        tail.append(line)
        if len(tail) > 200:
            tail.pop(0)
    returncode = process.wait()

if returncode != 0:
    print('\n===== BOOTSTRAP FAILURE TAIL =====', flush=True)
    print(''.join(tail), flush=True)
    print('Resolved GitHub commit:', resolved_sha, flush=True)
    print('Full bootstrap log:', LOG, flush=True)
    raise RuntimeError(f'Dialforge bootstrap exited with code {returncode}. Exact output is shown above.')

if not REPORT.exists():
    raise RuntimeError(f'Bootstrap returned success but report is missing: {REPORT}')
print('\n=== DIALFORGE BENCHMARK COMPLETE ===', flush=True)
print('Source commit:', resolved_sha, flush=True)
display(HTML(REPORT.read_text(encoding='utf-8')))


### Output files
- `/content/dialforge-benchmark/dialforge-benchmark-report.html`
- `/content/dialforge-benchmark/dialforge-benchmark-report.json`
- `/content/dialforge-bootstrap.log`

The launcher prints the exact Git commit used. It will refuse to run a stale v3 bootstrap.